# 🎓 Student Placement Prediction
**Dataset:** 1,00,000 rows · 8 features  
**Pipeline:** EDA → Preprocessing → 5-Model Comparison → Best Model Evaluation → Predict


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing     import StandardScaler
from sklearn.pipeline          import make_pipeline
from sklearn.metrics           import (accuracy_score, classification_report,
                                        confusion_matrix, roc_curve, roc_auc_score)
from sklearn.linear_model      import LogisticRegression
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm               import SVC
import joblib

sns.set_theme(style='whitegrid', palette='Set2')
print('Libraries loaded ✓')


In [ ]:
import os
print(os.listdir('/kaggle/input/datasets'))

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/sahil2604singh'))

## 2. Load Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/sahil2604singh/student-placement-dataset/placement.csv')
print('Shape:', df.shape)
print()
df.head(10)


In [ ]:
print(df.info())
print()
print(df.describe().round(2))


In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Placement counts:')
print(df['placement'].value_counts())
print(f'Placed %: {df["placement"].mean()*100:.1f}%')


## 3. Exploratory Data Analysis

In [ ]:
# Class balance bar chart
fig, ax = plt.subplots(figsize=(5,3))
counts = df['placement'].value_counts().rename({0:'Not Placed',1:'Placed'})
bars = counts.plot(kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='white', ax=ax)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x()+0.35, p.get_height()+200), fontsize=10)
ax.set_title('Placement Distribution', fontsize=13)
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
# Feature distributions by placement
features = ['cgpa','iq','internships','projects','communication_skill','backlogs','extra_curricular']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
colors = {0:'#e74c3c', 1:'#2ecc71'}
labels = {0:'Not Placed', 1:'Placed'}

for i, feat in enumerate(features):
    ax = axes[i//4][i%4]
    for lbl, grp in df.groupby('placement'):
        grp[feat].plot(kind='hist', ax=ax, alpha=0.6, bins=25,
                       label=labels[lbl], color=colors[lbl])
    ax.set_title(feat.replace('_',' ').title(), fontsize=11)
    ax.legend(fontsize=8)

axes[1][3].axis('off')
plt.suptitle('Feature Distributions by Placement Outcome', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# Box plots
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for i, feat in enumerate(features):
    ax = axes[i//4][i%4]
    df.boxplot(column=feat, by='placement', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red'))
    ax.set_title(feat.replace('_',' ').title(), fontsize=11)
    ax.set_xlabel('Placement (0=No, 1=Yes)')

axes[1][3].axis('off')
plt.suptitle('Box Plots by Placement', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(9,6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout(); plt.show()


## 4. Feature Engineering & Preprocessing

In [ ]:
FEATURES = ['cgpa','iq','internships','projects',
            'communication_skill','backlogs','extra_curricular']

# Composite score
df['composite_score'] = (
    df['cgpa']/10                * 0.35 +
    df['iq']/160                 * 0.20 +
    df['communication_skill']/10 * 0.25 +
    df['internships']/3          * 0.10 +
    df['projects']/5             * 0.10
)
FEATURES.append('composite_score')

X = df[FEATURES]
y = df['placement']

print('Features:', FEATURES)
print('X shape :', X.shape)


In [ ]:
# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train_sc.shape}')
print(f'Test  : {X_test_sc.shape}')


## 5. Train & Compare 5 Models (Stratified 5-Fold CV)
> Note: SVM is skipped on large data (too slow) — use the 4 tree/linear models.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Logistic Regression' : LogisticRegression(max_iter=1000),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=12,
                                                    random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150, learning_rate=0.08,
                                                        max_depth=5, random_state=42),
}

results = {}
for name, model in MODELS.items():
    print(f'Training {name}...')
    scores = cross_val_score(model, X_train_sc, y_train,
                              cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'  CV Acc = {scores.mean():.4f} ± {scores.std():.4f}')


In [ ]:
# Bar chart comparison
names = list(results.keys())
means = [results[n]['mean'] for n in names]
stds  = [results[n]['std']  for n in names]

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(names, means, xerr=stds,
               color=sns.color_palette('Set2', len(names)),
               edgecolor='white', capsize=5)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('Cross-Validation Accuracy')
ax.set_title('Model Comparison — 5-Fold CV (1 Lakh Dataset)', fontsize=13)
for bar, val in zip(bars, means):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout(); plt.show()


## 6. Best Model — Full Evaluation on Test Set

In [ ]:
best_name  = max(results, key=lambda n: results[n]['mean'])
best_model = MODELS[best_name]
print(f'Best model : {best_name}')
print(f'CV Accuracy: {results[best_name]["mean"]:.4f}')

best_model.fit(X_train_sc, y_train)
y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:, 1]

print(f'\nTest Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Test ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not Placed','Placed']))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Placed','Placed'],
            yticklabels=['Not Placed','Placed'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='#2ecc71', lw=2, label=f'{best_name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()


## 7. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
    imp.plot(kind='barh', figsize=(8,5), color='#3498db', edgecolor='white')
    plt.title(f'Feature Importance — {best_name}', fontsize=12)
    plt.xlabel('Importance')
    plt.tight_layout(); plt.show()
else:
    coefs = pd.Series(np.abs(best_model.coef_[0]), index=FEATURES).sort_values()
    coefs.plot(kind='barh', figsize=(8,5), color='#9b59b6', edgecolor='white')
    plt.title(f'Feature Coefficients — {best_name}', fontsize=12)
    plt.tight_layout(); plt.show()


## 8. Save Model & Scaler

In [ ]:
joblib.dump(best_model, 'placement_model.pkl')
joblib.dump(scaler,     'placement_scaler.pkl')
print('Saved: placement_model.pkl & placement_scaler.pkl ✓')


## 9. Predict on a New Student

In [ ]:
def predict_placement(cgpa, iq, internships, projects,
                      communication_skill, backlogs, extra_curricular):
    composite = (cgpa/10*0.35 + iq/160*0.20 +
                 communication_skill/10*0.25 +
                 internships/3*0.10 + projects/5*0.10)
    row = pd.DataFrame([{
        'cgpa': cgpa, 'iq': iq,
        'internships': internships, 'projects': projects,
        'communication_skill': communication_skill,
        'backlogs': backlogs, 'extra_curricular': extra_curricular,
        'composite_score': composite
    }])[FEATURES]
    row_sc = scaler.transform(row)
    pred   = best_model.predict(row_sc)[0]
    prob   = best_model.predict_proba(row_sc)[0][1]
    print(f'Prediction           : {"✅ PLACED" if pred==1 else "❌ NOT PLACED"}')
    print(f'Placement Probability: {prob:.2%}')

# Example student
predict_placement(cgpa=7.5, iq=130, internships=2, projects=4,
                  communication_skill=8.0, backlogs=0, extra_curricular=3)
